# Goal

To see how different optimization algorithms update weights of a model. The algorithms used will be:
1. Batch Gradient descent
2. Mini-batch gradient descent
3. Gradient with momentum (with mini-batch)
4. RMSprop (with mini-batch)
5. Adam optimizier (with mini-batch)

My theory is that first algorithm will be slow but the updates will have medium amount of noise. Second algorithm will be noiser but faster. The third should have low noise and be as fast as second, fourth should have roughly as much noise as third and be as fast while the last should have least noise and be as fast as second

In [11]:
# Imports
import numpy as np
import numpy.typing as npt
rng = np.random.default_rng(121)


## Preparations

The code below creates a synthetic dataset that I will use to visualize the optimizations. It will also have implementations for forward prop, backprop, cost function and gradient check to confirm implementation of backprop

In [7]:
# Generate synthetic data
synthetic_w_1 = rng.random() * 100
synthetic_w_2 = rng.random() * 100

m = 128
n = 2

synthetic_w = np.array([[synthetic_w_1, synthetic_w_2]])
noise = rng.random(m) * 0.1
X = rng.random((n, m)) * 100
Y = np.matmul(synthetic_w, X) + noise

In [22]:
def f_x(
    X: npt.NDArray,
    w: npt.NDArray,
    b: float
): 
    """ 
    Function for getting prediction from model
    
    Args:
        X (ndarray) : a (2, m) array with m training examples
        w (ndarray) : a (1, 2) array with weights for making prediction
        b (scalar) : bias for model
        
    Returns:
        Y_pred (ndarray): a (1,m) array with the models predictions
        Z (ndarray): cached Z value
    """
    Z = np.matmul(w, X) + b
    A = np.maximum(0, Z)
    return (A, Z)

In [23]:
def J(
    X: npt.NDArray,
    Y: npt.NDArray,
    w: npt.NDArray,
    b: float
):
    """ 
    Function to get cost of parameters in the dataset
    
    Args:
        X (ndarray): a (2, m) array with m training examples
        Y (ndarray): a (1, m) array with target outputs
        w (ndarray): a (1, 2) array with weights
        b (scalar): bias for the model
        
    Returns:
        cost (scalar): cost of weights for dataset
    """
    m = X.shape[1]
    Y_pred, _ = f_x(X=X, w=w, b=b)
    
    return np.sum((Y_pred - Y) ** 2) / (2 * m)
    

In [36]:
def backprop(
    X: npt.NDArray,
    Y: npt.NDArray,
    w: npt.NDArray,
    b: float
):
    """ 
    Returns derivatives of the weights and bias
    
    Args:
        X (ndarray): a (2, m) array with m training examples
        Y (ndarray): a (1, m) array with m target outputs
        w (ndarray): a (1,2) array with weights of the model
        b (scalar): bias of the model
        
    Returns:
        dw (ndarray): a (1,2) array with derivatives of the weights of the model
        db (scalar): derivative of bias
    """
    m = X.shape[1]
    Y_pred, Z = f_x(X=X, w=w, b=b)
    dA = Y_pred - Y
    dZ = np.where(Z < 0, 0, dA)
    dw = np.matmul(dZ, X.T) / m
    db = np.sum(dZ, axis=1)[0] / m
    
    return (dw, db)

In [41]:
# Confirming backprop implementation
def grad_check(
    X: npt.NDArray,
    Y: npt.NDArray,
    w: npt.NDArray,
    b: float
):
    """ 
    Checks whether derivatives returned from backprop and approximated derivatives are close enough
    
    Args:
        X (ndarray): a (2, m) array with m training examples
        Y (ndarray): a (1, m) array with m target outputs
        w (ndarray): a (1,2) array with weights of the model
        b (scalar): bias used by model
        
    Returns:
        similarity (scalar): measure of how close calculated and approximate derivatives are
        dtheta (ndarray): concatenated calculated derivatives
        dtheta_approx (ndarray): concatenated approximated derivatives
    """
    num_params = 3
    
    dw, db = backprop(X=X, Y=Y, w=w, b=b)
    theta = np.array([w[0][0], w[0][1], b])
    dtheta = np.array([dw[0][0], dw[0][1], db])
    dtheta_approx = np.zeros((num_params,))
    
    epsilon = 1e-7
    
    for i in range(num_params):
        theta_right = theta.copy()
        theta_right[i] += epsilon
        theta_left = theta.copy()
        theta_left[i] -= epsilon
        J_theta_right = J(X=X, Y=Y, w=np.array([[theta_right[0], theta_right[1]]]), b=theta_right[2])
        J_theta_left = J(X=X, Y=Y, w=np.array([[theta_left[0], theta_left[1]]]), b=theta_left[2])
        
        dtheta_approx[i] = (J_theta_right - J_theta_left) / (2 * epsilon)
        
    difference = np.linalg.norm(dtheta_approx - dtheta)
    similarity = difference / (np.linalg.norm(dtheta_approx) + np.linalg.norm(dtheta))
    return (similarity, dtheta, dtheta_approx)

In [44]:
initial_w = rng.random((1, 2))
similarity, dtheta, dtheta_approx = grad_check(X=X, Y=Y, w=initial_w, b=0)

print(f"Similarity between dtheta and dtheta approx is: {similarity}")
print(dtheta)
print(dtheta_approx)

assert similarity <= 1e-7 # If it fails this it is a bad implementation of backprop

Similarity between dtheta and dtheta approx is: 2.5724702768754786e-08
[-315044.48491426 -324383.26187098   -5462.43598178]
[-315044.50365901 -324383.26627016   -5462.44904399]


## Batch Gradient Descent